# [파이프라인 실습] AI City Challenge 2025 · Track 4 — 피시아이 도로 객체 탐지
## 우승팀(SmartVision) Inference 재현 — Colab 실습

이 노트북은 2025 AI City Challenge **Track 4(피시아이 카메라 도로 객체 탐지)** 우승팀 **SmartVision** 의 추론 과정을 Colab에서 따라가 봅니다.

- 검출기: **YOLO11**(우승팀은 YOLO11m을 사용) — 입력 **960×960**, conf **0.3**, NMS IoU **0.45**
- 클래스(FishEye8K, 5종): Bus, Bike, Car, Pedestrian, Truck

**사용법: 위에서부터 셀을 차례대로 실행(▶)하기만 하면 됩니다.** 이미지 업로드나 별도 설정은 필요 없습니다.

> **먼저 GPU를 켜세요**: 상단 메뉴 `런타임 → 런타임 유형 변경 → 하드웨어 가속기 → GPU(T4)` 선택 후 저장.


## 1) 환경 설치

필요한 라이브러리를 설치합니다. 1~2분 걸릴 수 있어요.


In [ ]:
!pip -q install ultralytics datasets
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| GPU 사용 가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


## 2) 모델 로드와 크기 확인

먼저 우승팀이 사용한 검출기 계열(**YOLO11m**)을 내려받아 **모델 크기(파라미터 수·연산량·파일 용량)** 를 직접 확인합니다. `YOLO(...)`는 가중치가 없으면 자동으로 내려받으니 그냥 실행하면 됩니다.


In [ ]:
from ultralytics import YOLO
import os

# 우승팀이 쓴 검출기 계열(YOLO11m) 가중치 자동 다운로드
m11 = YOLO('yolo11m.pt')
print(f'YOLO11m 파일 용량: {os.path.getsize("yolo11m.pt")/1e6:.1f} MB')

# 모델 크기 확인 — 레이어 수, 파라미터 수, GFLOPs(연산량)를 출력합니다
m11.info()


### FishEye8K로 학습된 가중치 내려받기 (실제 5클래스 검출용)

방금 받은 `yolo11m.pt`는 일반 객체(COCO 80종)용이라 그대로는 피시아이 5종에 딱 맞지 않습니다. 그래서 **FishEye8K로 학습된 공개 가중치**를 내려받아 사용합니다. 아래 코드는 대회 주최 측이 공개한 **YOLO11n FishEye8K 가중치**를 자동으로 받습니다.

```{admonition} 우승팀(SmartVision)의 가중치는 어디서 받나
:class: note

우승팀 가중치는 그들의 저장소 README **"2. Download Pretrained Model"** 항목의 OneDrive 링크로 배포됩니다.

- 저장소: https://github.com/vnptai/AICITY2025_track4
- 가중치(OneDrive): https://1drv.ms/u/c/3f64f78089759ca1/EXtAbIvc1N5FtYTKH18T5bcBoLbjzCYnXK7fGAMbM6PXPg?e=eHNeAC

압축 안에는 `yolo11m-obj365.pt`(Objects365 사전학습), `CO_DETR.pth`(증류용 교사 모델), `yolo11m.engine`(TensorRT 엔진)이 들어 있습니다. 최종 검출 모델은 학습 후 50 epoch 체크포인트를 `yolo11m.pt`로 사용합니다.

받은 뒤에는 위 `WEIGHTS` 경로를 그 파일로 바꾸면 동일한 코드로 동작합니다(예: `WEIGHTS = '/content/yolo11m.pt'`). 단, **OneDrive는 브라우저 다운로드가 필요**해 `wget`/`urllib`로 Colab에서 자동으로 받기는 어렵습니다. 직접 내려받아 Colab에 업로드하거나 드라이브에 올린 뒤 경로를 지정하세요.
```


In [ ]:
import urllib.request, os

# FishEye8K로 학습된 공개 가중치(대회 주최측 baseline, YOLO11n) 자동 다운로드
URL = 'https://raw.githubusercontent.com/MoyoG/FishEye8K/main/challenge_iccv_2025/models/yolo11n_fisheye8k.pt'
WEIGHTS = 'yolo11n_fisheye8k.pt'
urllib.request.urlretrieve(URL, WEIGHTS)
print(f'내려받음: {WEIGHTS} ({os.path.getsize(WEIGHTS)/1e6:.1f} MB)')

# 앞으로 이 모델로 추론합니다
model = YOLO(WEIGHTS)
print('클래스:', model.names)   # 0 Bus, 1 Bike, 2 Car, 3 Pedestrian, 4 Truck

# 참고: 큰 모델(m)과 작은 모델(n)의 파라미터 차이 — 엣지일수록 작은 모델을 선호합니다
pm = sum(p.numel() for p in m11.model.parameters())/1e6
pn = sum(p.numel() for p in model.model.parameters())/1e6
print(f'YOLO11m ≈ {pm:.1f}M 파라미터 vs YOLO11n ≈ {pn:.1f}M 파라미터')


## 3) FishEye8K 샘플 이미지 자동 내려받기

업로드 없이, 공개 **FishEye8K 데이터셋**에서 샘플 이미지 3장을 자동으로 받아옵니다. 그냥 실행하세요.

- 데이터셋(HuggingFace): https://huggingface.co/datasets/Voxel51/fisheye8k
- 원본 저장소: https://github.com/MoyoG/FishEye8K


In [ ]:
from datasets import load_dataset
import os
os.makedirs('samples', exist_ok=True)

# 공개 FishEye8K 데이터셋에서 샘플 3장 받아 저장
ds = load_dataset('Voxel51/fisheye8k', split='train', streaming=True)
paths = []
for i, ex in zip(range(3), ds):
    p = f'samples/fisheye_{i}.jpg'
    ex['image'].convert('RGB').save(p)
    paths.append(p)
print('저장된 샘플:', paths)

IMG = paths[0]   # 첫 번째 샘플로 데모합니다

# (선택) 내 피시아이/CCTV 이미지를 쓰고 싶다면 아래 두 줄의 주석을 해제하세요.
# from google.colab import files; up = files.upload(); IMG = list(up.keys())[0]


## 4) 추론 실행 (우승팀과 동일한 설정)

입력 960, conf 0.3, IoU 0.45로 추론하고 결과를 그림으로 봅니다.


In [ ]:
import matplotlib.pyplot as plt

results = model.predict(IMG, imgsz=960, conf=0.3, iou=0.45, verbose=False)
annotated = results[0].plot()[:, :, ::-1]   # BGR -> RGB

plt.figure(figsize=(12, 8))
plt.imshow(annotated); plt.axis('off')
plt.title('FishEye8K detection (YOLO11, imgsz=960, conf=0.3, iou=0.45)')
plt.show()

# 검출 요약
for b in results[0].boxes:
    cls = model.names[int(b.cls)]
    print(f'{cls:12s} conf={float(b.conf):.2f}')


## 5) 추론 속도 측정 — "엣지 실시간" 요건 감 잡기

Track 4는 Jetson AGX Orin에서 **FPS > 10** 을 요구했습니다. Colab GPU 기준 지연/FPS를 재봅니다.


In [ ]:
import time

for _ in range(3):   # 워밍업
    model.predict(IMG, imgsz=960, conf=0.3, iou=0.45, verbose=False)

N = 30
t0 = time.time()
for _ in range(N):
    model.predict(IMG, imgsz=960, conf=0.3, iou=0.45, verbose=False)
dt = (time.time() - t0) / N
print(f'평균 지연: {dt*1000:.1f} ms/장  ->  처리량: {1/dt:.1f} FPS')
print('참고: Colab GPU와 Jetson은 다릅니다. 실제 엣지 성능은 TensorRT 변환 후 측정해야 합니다.')


## 6) 엣지 최적화 — TensorRT 변환 (우승팀의 optimization 단계)

우승팀은 학습한 모델을 **TensorRT 엔진(FP16/INT8)** 으로 변환해 Jetson에서 실시간을 달성했습니다. 다만 이 경량화/엔진 빌드는 본래 **배포 대상 하드웨어(예: Jetson)** 에서 하는 단계이고, 이번 실습에서 직접 돌려볼 필요는 없습니다.

```{admonition} Colab(T4)에서는 실행하지 않습니다
:class: warning

Colab의 TensorRT 버전(AutoUpdate로 설치되는 11.x)과 export 코드가 호환되지 않아 다음과 같은 오류가 납니다.

    AttributeError: type object '...BuilderFlag' has no attribute 'FP16'

이는 모델·코드 문제가 아니라 **Colab 환경의 TensorRT 버전 충돌**입니다. 엔진 변환은 실제 타깃 장비(Jetson 등)에서 그 장비의 TensorRT로 수행하는 것이 맞으므로, 여기서는 실행하지 않고 개념만 짚고 넘어갑니다.
```

참고로, 타깃 장비에서 변환·추론하는 코드는 다음과 같습니다(실행용 아님, 개념 참고).

    # 타깃 장비(GPU+호환 TensorRT)에서:
    engine_path = model.export(format='engine', imgsz=960, half=True)  # FP16 엔진 생성
    trt = YOLO(engine_path)
    trt.predict(IMG, imgsz=960, conf=0.3, iou=0.45)

우승팀은 여기에 더해 INT8 양자화와 CUDA 전·후처리까지 직접 구현해 Jetson AGX Orin에서 FPS > 10을 만족시켰습니다. 즉 **정확도(F1)와 속도(FPS)를 동시에 맞추는 것**이 이 트랙의 최종 관문입니다.


---

# 심화 실습 — CCTV 영상 분석 & 안전 위반(ROI 침입) 탐지

지금까지는 "이미지 한 장"을 추론했습니다. 실제 CCTV 안전 모니터링은 **영상**을 보고 **위반 이벤트**를 잡아야 합니다. 아래 네 단계로 확장합니다. 역시 위에서부터 실행만 하면 됩니다.

1. 이미지 → **영상(연속 프레임)** 파이프라인
2. **객체 추적(ByteTrack)** — 대상에 ID 부여
3. **ROI(금지구역) 침입 탐지** — 규칙 레이어
4. **시간적 안정화** — 연속 N프레임 조건으로 오탐 줄이기

안전 위반 시나리오는 **"금지구역에 사람·차량이 들어와 일정 시간 머무름"** 으로 잡습니다. 탐지 모델 자체는 "위반"을 모르므로, 탐지 결과 위에 **추적 + 규칙 + 시간 조건**을 얹어 위반을 정의하는 것이 핵심입니다.


## 7) 이미지 → 영상 파이프라인

FishEye8K는 여러 영상에서 뽑은 프레임 모음입니다. 그 앞부분은 **같은 카메라의 연속 프레임**이라, 이어 붙이면 실제 CCTV 같은 짧은 클립이 됩니다. 60장을 받아 클립으로 만듭니다.


In [ ]:
!pip -q install -U lap   # 추적기(ByteTrack)에 필요
import os, cv2
from datasets import load_dataset

os.makedirs('clip', exist_ok=True)
NUM = 60
ds = load_dataset('Voxel51/fisheye8k', split='train', streaming=True)
frames = []
for i, ex in zip(range(NUM), ds):
    p = f'clip/f{i:03d}.jpg'
    ex['image'].convert('RGB').save(p)
    frames.append(p)
print(f'{len(frames)} 프레임 확보 (같은 카메라의 연속 장면)')

H, W = cv2.imread(frames[0]).shape[:2]
vw = cv2.VideoWriter('input.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 10, (W, H))
for fp in frames:
    vw.write(cv2.imread(fp))
vw.release()
print('입력 클립 저장 완료: input.mp4  해상도', (W, H))


## 8) 객체 추적(ByteTrack) — 대상에 ID 부여

"위반"은 보통 *같은 대상이 얼마 동안* 같은 시간 조건을 포함합니다. 따라서 프레임마다 같은 객체에 **같은 ID**를 붙이는 추적이 먼저 필요합니다. `model.track(..., persist=True)` 한 줄이면 됩니다(검출=딥러닝 + 추적=칼만필터 결합).


In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO(WEIGHTS)          # 추적 상태를 새로 시작하기 위해 다시 로드
track_data = []               # 프레임별 (id, class, box) 저장 → 뒤 단계에서 재사용
tracked_vis = []
for fp in frames:
    img = cv2.imread(fp)
    res = model.track(img, persist=True, tracker='bytetrack.yaml',
                      imgsz=960, conf=0.3, iou=0.45, verbose=False)
    b = res[0].boxes
    if b is not None and b.id is not None:
        ids  = b.id.int().tolist()
        cls  = b.cls.int().tolist()
        xyxy = b.xyxy.tolist()
    else:
        ids, cls, xyxy = [], [], []
    track_data.append({'ids': ids, 'cls': cls, 'xyxy': xyxy})
    tracked_vis.append(res[0].plot())   # 박스+ID가 그려진 프레임

uniq = {t for d in track_data for t in d['ids']}
print('총 고유 추적 ID 수:', len(uniq))

# 추적 결과 영상 저장
vw = cv2.VideoWriter('tracked.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 10, (W, H))
for f in tracked_vis: vw.write(f)
vw.release()

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 8)); plt.imshow(tracked_vis[len(frames)//2][:, :, ::-1])
plt.axis('off'); plt.title('ByteTrack: 박스 + 추적 ID'); plt.show()


## 9) ROI(금지구역) 정의와 침입 규칙

감시할 **금지구역**을 다각형으로 정의합니다(여기서는 화면 비율로 지정해 어떤 영상에도 동작). 각 대상의 **바닥 지점(박스 아래 중앙)** 이 이 구역 안에 들어오면 "침입 후보"로 봅니다. point-in-polygon 판정은 `cv2.pointPolygonTest` 한 줄입니다.


In [ ]:
import numpy as np, cv2
import matplotlib.pyplot as plt

# 금지구역 다각형 (화면 비율로 정의)
ROI = np.array([[int(W*0.30), int(H*0.45)], [int(W*0.70), int(H*0.45)],
                [int(W*0.80), int(H*0.95)], [int(W*0.20), int(H*0.95)]], np.int32)

def in_roi(x1, y1, x2, y2):
    gx, gy = int((x1 + x2) / 2), int(y2)        # 바닥 중앙점
    return cv2.pointPolygonTest(ROI, (gx, gy), False) >= 0

# ROI를 한 프레임 위에 그려서 확인
vis = cv2.imread(frames[0]).copy()
cv2.polylines(vis, [ROI], True, (0, 200, 255), 3)
plt.figure(figsize=(10, 8)); plt.imshow(vis[:, :, ::-1]); plt.axis('off')
plt.title('감시할 금지구역(ROI)'); plt.show()


## 10) 시간적 안정화 — 연속 N프레임 조건으로 위반 확정

한 프레임만 보고 경고하면 오탐이 많습니다. **같은 ID가 ROI 안에 연속 K프레임 이상 머무를 때만** 위반으로 확정합니다(디바운싱). 확정 위반은 빨간 박스로 표시하고, 이벤트(프레임·ID·클래스)를 기록합니다.


In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt

K = 5                      # 연속 K프레임 이상 체류 시 위반 확정
TARGET = None              # None=모든 클래스. 예: {'Pedestrian'} 으로 제한 가능

inside_cnt, confirmed, events, out_frames = {}, set(), [], []
for fi, fp in enumerate(frames):
    img = cv2.imread(fp)
    cv2.polylines(img, [ROI], True, (0, 200, 255), 2)
    cur = set()
    d = track_data[fi]
    for tid, c, box in zip(d['ids'], d['cls'], d['xyxy']):
        name = model.names[c]
        if TARGET and name not in TARGET:
            continue
        x1, y1, x2, y2 = box
        if in_roi(x1, y1, x2, y2):
            cur.add(tid)
            inside_cnt[tid] = inside_cnt.get(tid, 0) + 1
            if inside_cnt[tid] == K and tid not in confirmed:
                confirmed.add(tid)
                events.append({'frame': fi, 'id': tid, 'class': name})
        violation = (tid in confirmed) and in_roi(x1, y1, x2, y2)
        color = (0, 0, 255) if violation else (0, 180, 0)
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        label = f'{name} id{tid}' + ('  VIOLATION' if violation else '')
        cv2.putText(img, label, (int(x1), int(y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    for tid in list(inside_cnt):       # ROI를 벗어나면 카운터 초기화
        if tid not in cur:
            inside_cnt[tid] = 0
    out_frames.append(img)

# 결과 영상 저장
vw = cv2.VideoWriter('violations.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 10, (W, H))
for f in out_frames: vw.write(f)
vw.release()

# 이벤트 로그 출력
print(f'확정된 위반 {len(events)}건')
for e in events:
    print(f"  - 프레임 {e['frame']:>3} | id {e['id']} | {e['class']} 금지구역 침입")

# 핵심 장면 3장 보기 (위반 발생 전후)
if events:
    f0 = events[0]['frame']
    picks = [max(0, f0 - 6), f0, min(len(frames) - 1, f0 + 6)]
else:
    picks = [0, len(frames)//2, len(frames) - 1]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, idx in zip(axes, picks):
    ax.imshow(out_frames[idx][:, :, ::-1]); ax.axis('off'); ax.set_title(f'frame {idx}')
plt.show()


이 패턴(영상 루프 → 추적 → 규칙 → 시간 안정화)은 ROI 침입 외에도 **배회 감지·역주행·근접 위험·혼잡도** 등 다양한 안전 규칙으로 확장할 수 있습니다. ROI 좌표(`ROI`), 연속 프레임 임계값(`K`), 대상 클래스(`TARGET`)만 바꿔 실험해 보세요.


## 정리

- 우승 비결은 *더 큰 모델*이 아니라 **(Objects365 사전학습) + (CO-DETR 교사 distillation·pseudo-labeling) + (합성 피시아이 증강) + (TensorRT 최적화)** 의 조합이었습니다.
- 이 노트북의 라이브 데모는 바로 받을 수 있는 **FishEye8K 공개 가중치(YOLO11n)** 를 썼습니다. 우승팀의 **YOLO11m 파인튜닝 가중치**로 바꾸면 같은 코드로 더 높은 성능을 재현할 수 있습니다(`WEIGHTS` 경로만 교체).

### 참고 링크
- 우승팀 SmartVision 코드: https://github.com/vnptai/AICITY2025_track4
- 우승팀 가중치(OneDrive, README의 'Download Pretrained Model'): https://1drv.ms/u/c/3f64f78089759ca1/EXtAbIvc1N5FtYTKH18T5bcBoLbjzCYnXK7fGAMbM6PXPg?e=eHNeAC
- 상위팀 코드 모음: https://github.com/NVIDIAAICITYCHALLENGE/2025AICITY_Code_From_Top_Teams
- Track 4 페이지: https://www.aicitychallenge.org/2025-track4/
- FishEye8K 데이터셋: https://github.com/MoyoG/FishEye8K
- YOLO11 문서: https://docs.ultralytics.com/models/yolo11/
- 심화 실습에서는 추적(ByteTrack) + ROI 침입 규칙 + 시간 안정화로 **CCTV 안전 위반 이벤트**까지 만들어 봤습니다.
